## This is my work for problems 7-10 of inference homework

In [108]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [109]:
ames_prices = pd.read_csv('./data/ames_prices.csv')
ames_prices.head()

,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


In [110]:
ames_prices1 = pd.get_dummies(ames_prices, columns=['Central.Air'], dtype=int, drop_first=True)

### Question 7

#### 7a.

In [111]:
central_air = (ames_prices1['Central.Air_Y'] == 1).mean()
print(f'Proportion of homes WITH central air: {central_air:.2f}')

Proportion of homes WITH central air: 0.93


#### 7b.

In [112]:
## Bootstrapping:
np.random.seed(53)

R = 2_000

ca = []

for t in range(R):
    ## Resample:
    idx = np.random.randint(0, len(ames_prices1), len(ames_prices1))
    ## Evaluate outcomes:
    ca_t = (ames_prices1['Central.Air_Y'].iloc[idx] == 1).mean()
    
    ## Store results:
    ca.append(ca_t)

print(f'I set the seed to 53 and the number of bootstrap replications to {R}')

I set the seed to 53 and the number of bootstrap replications to 2000


#### 7c.

In [113]:
std_error_ca = np.std(ca, ddof=1)
print(std_error_ca)

0.004622098982113707


#### 7d.

In [114]:
confidence_interval_ninety = (central_air - 1.645 * std_error_ca, central_air + 1.645 * std_error_ca)
print(f'90% CI: {confidence_interval_ninety[0]:.3f}, {confidence_interval_ninety[1]:.3f}')

confidence_interval_ninetyfive = (central_air - 1.96 * std_error_ca, central_air + 1.96 * std_error_ca)
print(f'95% CI: {confidence_interval_ninetyfive[0]:.3f}, {confidence_interval_ninetyfive[1]:.3f}')

confidence_interval_ninetynine = (central_air - 2.576 * std_error_ca, central_air + 2.576 * std_error_ca)
print(f'99% CI: {confidence_interval_ninetynine[0]:.3f}, {confidence_interval_ninetynine[1]:.3f}')

90% CI: 0.926, 0.941
95% CI: 0.924, 0.942
99% CI: 0.921, 0.945


#### 7e.

Based on the above confidence intervals we would reject the null hypothesis that $p=0.9$ as none of the confidence intervals cointain $0.9$. 

### Question 8

In [115]:
area_transformed = np.arcsinh(ames_prices['area'])
price_transformed = np.arcsinh(ames_prices['price'])

price_area_corr = np.corrcoef(area_transformed, price_transformed)
p_a_corr = price_area_corr[0,1]
print(f'The Peasron correlation is: {np.round(p_a_corr, 2)}')


The Peasron correlation is: 0.72


#### 8a.

In [116]:
## Bootstrapping:
np.random.seed(53)

R = 2_000

pa_corr = []

for t in range(R):
    ## Resample:
    idx = np.random.randint(0, len(ames_prices), len(ames_prices))
    ## Evaluate outcomes:
    a_t = np.arcsinh(ames_prices['area'].iloc[idx])
    p_t = np.arcsinh(ames_prices['price'].iloc[idx])

    pa_corr_t = np.corrcoef(a_t, p_t)
    
    ## Store results:
    pa_corr.append(pa_corr_t)

print(f'I set the seed to 53 and the number of bootstrap replications to {R}')

I set the seed to 53 and the number of bootstrap replications to 2000


#### 8.b

In [117]:
std_error_pa_corr = np.std(pa_corr, ddof=1)
print(std_error_pa_corr)

0.13866078860716088


#### 8c.

In [118]:
confidence_interval_ninety = (p_a_corr - 1.645 * std_error_pa_corr, p_a_corr + 1.645 * std_error_pa_corr)
print(f'90% CI: {confidence_interval_ninety[0]:.3f}, {confidence_interval_ninety[1]:.3f}')

confidence_interval_ninetyfive = (p_a_corr - 1.96 * std_error_pa_corr, p_a_corr + 1.96 * std_error_pa_corr)
print(f'95% CI: {confidence_interval_ninetyfive[0]:.3f}, {confidence_interval_ninetyfive[1]:.3f}')

confidence_interval_ninetynine = (p_a_corr - 2.576 * std_error_pa_corr, p_a_corr + 2.576 * std_error_pa_corr)
print(f'99% CI: {confidence_interval_ninetynine[0]:.3f}, {confidence_interval_ninetynine[1]:.3f}')

90% CI: 0.495, 0.951
95% CI: 0.451, 0.995
99% CI: 0.366, 1.080


#### 8d.

Based on the confidence intervals above we would fail to reject the null at each level of significance that $p=0$. This is because at each significance level, the confidence interval does not contain $0$.

#### 8e.

They need to be paired together during resampling because each row of our data corresponds to one property that has a value for `area` and `price`. Thus if we resampled them not as a pair, we'd lose that relationship when calculating the correlation between them.

### Question 9

In [119]:
metabric = pd.read_csv("./data/metabric.csv")

mean_ttd = metabric['Overall Survival (Months)'].mean()
print(f'The mean overall time to death is {mean_ttd:.2f} months')

The mean overall time to death is 129.09 months


#### 9a.

In [120]:
## Bootstrapping:
np.random.seed(53)

R = 2_000

ttd = []

for t in range(R):
    ## Resample:
    idx = np.random.randint(0, len(metabric), len(metabric))
    ## Evaluate outcomes:
    ttd_t = metabric['Overall Survival (Months)'].iloc[idx].mean()
    
    ## Store results:
    ttd.append(ttd_t)

print(f'I set the seed to 53 and the number of bootstrap replications to {R}')

I set the seed to 53 and the number of bootstrap replications to 2000


#### 9b.

In [121]:
std_error_ttd= np.std(ttd, ddof=1)
print(std_error_ttd)

2.1201760823386357


#### 9c.

In [122]:
confidence_interval_ninety = (mean_ttd - 1.645 * std_error_ttd, mean_ttd + 1.645 * std_error_ttd)
print(f'90% CI: {confidence_interval_ninety[0]:.3f}, {confidence_interval_ninety[1]:.3f}')

confidence_interval_ninetyfive = (mean_ttd - 1.96 * std_error_ttd, mean_ttd + 1.96 * std_error_ttd)
print(f'95% CI: {confidence_interval_ninetyfive[0]:.3f}, {confidence_interval_ninetyfive[1]:.3f}')

confidence_interval_ninetynine = (mean_ttd - 2.576 * std_error_ttd, mean_ttd + 2.576 * std_error_ttd)
print(f'99% CI: {confidence_interval_ninetynine[0]:.3f}, {confidence_interval_ninetynine[1]:.3f}')

90% CI: 125.607, 132.582
95% CI: 124.939, 133.250
99% CI: 123.633, 134.556


#### 9d.

Based on the above confidence intervals, we would reject the null hypothesis that the mean recorded time to death is 120. This is because at each significance level, 120 is not included in the confidence interval.

#### 9e.

The estimated recorded time to death for this data was 129.09 months, or 10.76 years. The intervals say that if we resampled many times, 90%, 95, and 99% of the samples would include the true value. These intervals are 125.607 to 132.582 months, or 10.46 to 11.05 years, at a significance level of 0.10, 124.939 to 133.250 months, or 10.41 to 11.10 years, at a significance level of 0.05, and 123.633 to 134.556, or 10.30 to 11.21 years, at a significance level of 0.01.

But this estimate and the confidence interval does not estimate the unrestricted mean lifetime because it includes patients that are still alive. This means that the recorded time to death for patients that were still alive at the time they were recorded is the lowest possible time to death for that patient not the actual time to death like it is for patients that are deceased.

### Question 10

In [123]:
nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")
nhanes_filtered = nhanes[(nhanes['AgeInYearsAtScreening'] > 18) & (nhanes['SleepHoursWeekdaysOrWorkdays'].notna())]
mean_sleep = nhanes_filtered['SleepHoursWeekdaysOrWorkdays'].mean()

print(f'The mean sleep hours is {mean_sleep:.2f} hours')


The mean sleep hours is 7.61 hours


/var/folders/rf/v6lnf4j94_j1gq143nx7p8f80000gn/T/ipykernel_16568/4026229284.py:1: DtypeWarning: Columns (142) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")


#### 10a.

In [124]:
## Bootstrapping:
np.random.seed(53)

R = 2_000

avgsleep = []

for t in range(R):
    ## Resample:
    idx = np.random.randint(0, len(nhanes_filtered), len(nhanes_filtered))
    ## Evaluate outcomes:
    avgsleep_t = nhanes_filtered['SleepHoursWeekdaysOrWorkdays'].iloc[idx].mean()
    
    ## Store results:
    avgsleep.append(avgsleep_t)

print(f'I set the seed to 53 and the number of bootstrap replications to {R}')

I set the seed to 53 and the number of bootstrap replications to 2000


#### 10b.

In [125]:
std_error_avgsleep = np.std(avgsleep, ddof=1)
print(std_error_avgsleep)

0.023032582684499255


In [126]:
confidence_interval_ninety = (mean_sleep - 1.645 * std_error_avgsleep, mean_sleep + 1.645 * std_error_avgsleep)
print(f'90% CI: {confidence_interval_ninety[0]:.3f}, {confidence_interval_ninety[1]:.3f}')

confidence_interval_ninetyfive = (mean_sleep - 1.96 * std_error_avgsleep, mean_sleep + 1.96 * std_error_avgsleep)
print(f'95% CI: {confidence_interval_ninetyfive[0]:.3f}, {confidence_interval_ninetyfive[1]:.3f}')

confidence_interval_ninetynine = (mean_sleep - 2.576 * std_error_avgsleep, mean_sleep + 2.576 * std_error_avgsleep)
print(f'99% CI: {confidence_interval_ninetynine[0]:.3f}, {confidence_interval_ninetynine[1]:.3f}')

90% CI: 7.568, 7.643
95% CI: 7.560, 7.651
99% CI: 7.546, 7.665


#### 10d.

For every significance level we would reject the null hypothesis that the population average hours of sleep is 8. This is because every confidence interval at the matching significance levels does not contain 8.

#### 10e.
While it is the case that we rejected the null hypothesis that the mean hours of sleep per night is 8, our range of where we expect the true population value to fall between is around 7.5 and 7.6. This is not only a small range, but is only 0.4-0.5 away from 8. Practically the difference between 7.5/6 and 8 hours of sleep is not large, it was enough for 8 hours to be rejected as the true mean statistically.   